In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import tensorflow_hub as hub
import tensorflow_text as text

In [37]:
labelled_emails_df = pd.read_csv('../data/04_encoded_labelled_emails.csv')
labelled_emails_df.head()

,Date,From,To,Subject,X_From,X_To,Message,Entire-Message,DISC_rule,DISC_manual,DISC_final,D,I,S,C
0,"Fri, 25 Aug 2000 03:30:00 -0700 (PDT)",phillip.allen@enron.com,brad.mcsherry@enron.com,"Cc: [EMAIL], [EMAIL]",Phillip K Allen,Brad McSherry,"Brad,\nWith regard to Tori Kuykendall, I would...","Cc: [EMAIL], [EMAIL] Brad,\nWith regard to Tor...","['D', 'S']",[],"['D', 'S']",1,0,1,0
1,"Mon, 10 Jul 2000 06:54:00 -0700 (PDT)",phillip.allen@enron.com,al.pollard@enron.com,Re: Katy flatlands,Phillip K Allen,Al Pollard,"Al,\nI am not in good enough shape to ride a c...","Re: Katy flatlands Al,\nI am not in good enoug...","['D', 'I']",[],"['D', 'I']",1,1,0,0
2,"Tue, 24 Apr 2001 13:05:00 -0700 (PDT)",phillip.allen@enron.com,jsmith@austintx.com,Re: The Stage,Phillip K Allen,Jeff Smith <jsmith@austintx.com>,I just spoke to the insurance company. They ar...,Re: The Stage I just spoke to the insurance co...,['C'],[],['C'],0,0,0,1
3,"Fri, 9 Mar 2001 05:06:00 -0800 (PST)",phillip.allen@enron.com,stagecoachmama@hotmail.com,No Subject,Phillip K Allen,stagecoachmama@hotmail.com,"Lucy,\nHere is a rentroll for this week.\nWhat...","No Subject Lucy,\nHere is a rentroll for this ...",['D'],[],['D'],1,0,0,0
4,"Mon, 5 Mar 2001 07:21:00 -0800 (PST)",phillip.allen@enron.com,cbpres@austin.rr.com,"Cc: [EMAIL], [EMAIL]",Phillip K Allen,cbpres@austin.rr.com,"George,\nI am back in the office and ready to ...","Cc: [EMAIL], [EMAIL] George,\nI am back in the...","['I', 'S']",[],"['I', 'S']",0,1,1,0


In [38]:
labelled_emails_df.isnull().sum()

Date              0
From              0
To                0
Subject           0
X_From            0
X_To              0
Message           0
Entire-Message    0
DISC_rule         0
DISC_manual       0
DISC_final        0
D                 0
I                 0
S                 0
C                 0
dtype: int64

In [39]:
mlb = MultiLabelBinarizer(classes=['D', 'I', 'S', 'C'])
Y = mlb.fit_transform(labelled_emails_df['DISC_final']) # Learns the unique labels
mlb.classes_ # The labels learned in sorted order
Y = mlb.transform(labelled_emails_df['DISC_final'])

/Users/pkchoy/Portfo/disc-predictor-for-business/.business-disc/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) [' ', "'", ',', '[', ']'] will be ignored
  warnings.warn(


## Split

In [40]:
X_train_text, X_test_text, Y_train, Y_test = train_test_split(labelled_emails_df['Entire-Message'], Y, test_size=0.2, random_state=32)
print(type(X_train_text))

<class 'pandas.core.series.Series'>


## Feature Extraction: Generate BERT Embeddings from Entire-Message

In [33]:
# bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
# bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

# print(os.getcwd())
# print(os.listdir("tfhub_modules"))

bert_preprocess = hub.KerasLayer("../tfhub_modules/bert_preprocess", trainable=False)
bert_encoder = hub.KerasLayer("../tfhub_modules/bert_encoder", trainable=False)

In [34]:
# def get_bert_embeddings(texts):
#     text_inputs = tf.constant(texts)
#     tokenized_text = bert_preprocess(text_inputs)
#     outputs = bert_encoder(tokenized_text)
#     print(outputs.keys())
#     return outputs['pooled_output'].numpy()


# X_train = get_bert_embeddings(X_train_text.tolist())
# X_test = get_bert_embeddings(X_test_text.tolist())

# np.save("../embeddings/X_train_bert.npy", X_train)
# np.save("../embeddings/X_test_bert.npy", X_test)

# print("X_train", X_train)
# print("X_testX", X_test)

In [35]:
X_train = np.load("../embeddings/X_train_bert.npy")
X_test = np.load("../embeddings/X_test_bert.npy")

print(X_train)
print(X_test)
print(X_train.shape, X_test.shape)

[[-0.6314712  -0.52617776 -0.94768167 ... -0.5904847  -0.6615315
   0.77338326]
 [-0.7271016  -0.50922173 -0.966358   ... -0.8682175  -0.58734834
   0.53512746]
 [-0.6152391  -0.5291411  -0.97700727 ... -0.9766525  -0.6205291
   0.55183804]
 ...
 [-0.564998   -0.42946798 -0.92218417 ... -0.7160475  -0.611352
   0.61186284]
 [-0.77536887 -0.4544915  -0.96167237 ... -0.9183127  -0.4488551
   0.38754222]
 [-0.47860402 -0.5482809  -0.9917552  ... -0.96489906 -0.63556933
   0.18608356]]
[[-0.28295514 -0.25646207 -0.9408917  ... -0.86757624 -0.4368691
   0.29567564]
 [-0.17603873 -0.02102108  0.3163479  ...  0.04302573 -0.2302131
   0.31858164]
 [-0.6733913  -0.3732347  -0.5620626  ... -0.6899617  -0.67061347
   0.6852762 ]
 ...
 [ 0.00404818 -0.49441555 -0.9444312  ... -0.7161667  -0.4412066
   0.20382577]
 [-0.68039745 -0.5984991  -0.9904107  ... -0.9433675  -0.73913074
   0.53691906]
 [ 0.02135798 -0.35530376 -0.9654061  ... -0.942927   -0.4678314
   0.18492217]]
(2088, 768) (523, 768)
